# Using G2ELin for beginners : 
Learn to use the main blocks of the tool (Building a network from scratch or calling a preset network --> Power Flow (PandaPower) --> Modal analysis toolbox --> EMT time-domain simulations --> ROA analysis)

## 0. Initialisation and basic imports :

In [ ]:
# The basic imports :
%matplotlib inline
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

plt.rcParams["figure.dpi"] = 100
pd.set_option("display.precision", 4)
np.set_printoptions(precision=4, suppress=True)

# The imports related to the G2ELin library :
from g2elin_core.network.schema import (
    Bus, Line, Transformer, Load, DerUnit, Network, BusType, UnitType, GfmController,
)
from g2elin_core.network.validation import validate_network
from g2elin_core.network.topology import compute_topology_layout
from g2elin_core.powerflow import run_power_flow
from g2elin_core.pipeline import linearize_network
from g2elin_core.modal import analyze

#### Run-time toggles

Section 6 (nonlinear EMT simulation) and Section 7 (region-of-attraction
tracing) are the slowest cells in this notebook -- each is a full
*nonlinear* coupled-Newton time integration, not just linear algebra.
Both default to **off** so "Restart Kernel and Run All" finishes quickly;
flip either flag to `True` and re-run to actually exercise that section.

In [ ]:
RUN_EMT_SECTION = True  # Section 6: nonlinear EMT time-domain simulation
RUN_ROA_SECTION = False  # Section 7: region-of-attraction grid tracing

## 1. Creating the network : 
a) creating the skeleton : either
- Create a network from scratch
- Load a preset network (SMIB, CIGRE distribution network, WSCC transmission network)

b) editing default parameters of each element

c) validating the network 

### 1.1. Loading a preset network : 
Available network presets :
1. CIGRE distribution networks : 

    a) `cigre_islanded_1sm_1gfm_1gfl()`
    
    b) ..

2. WSCC transmission networks : 

    a) `wscc_3sm()`
    
    b) `wscc_1gfm_1gfl_1sm()`

    
3. SMIB configurations : 

In [ ]:
# printing all available network presets
from g2elin_core.network.presets import list_network_presets

print("Available network presets:")
for preset in list_network_presets():
    print(f"  - {preset}")

In [ ]:
# selecting a network preset to use
preset_name = "cigre_islanded_1sm_1gfm_1gfl"

# loading the network preset
network_preset = Network.from_preset(preset_name)

print(f"Network preset '{preset_name}' loaded successfully.")

In [ ]:
# Investigfating the network preset

# plotting the network topology
topology_layout = compute_topology_layout(network_preset)

In [ ]:
# Investigating elements parameters : 
# DER units parameters : 

# transformer parametrs : 

# loads parameters :

# lines parameters :

### 1.2. Building a network from scratch

#### The convention a network builder has to respect

Paraphrasing `network.md`'s Validation section, and matching
`Functions/network_form.m`'s original MATLAB convention:

- Every DER unit (`sm`/`gfm`/`gfl`/`infinite_bus`) sits on **its own
  private bus**, reached from the rest of the network through **exactly
  one transformer** — never a `Line`, never shared with another DER, never
  another transformer's `hv_bus`.
- A DER's own local consumption is its `p_cons_mw`/`q_cons_mvar` fields,
  not a separate `Load` on its private bus.
- **Exactly one** DER unit has `bus_type = slack`, and for modal
  analysis/EMT/ROA to work it should be an `sm` or `infinite_bus` (a
  `gfm`/`gfl` slack is accepted for power flow but not wired into
  `interconnect.network_assembly` yet — `validate_network()` flags this
  as a warning, not an error).
- The whole graph (buses + lines + transformers) must be **connected** —
  every bus must be reachable from the slack.

None of that is specific to any one topology *shape* — a schema-valid
network can be meshed or radial, tiny or large, one DER or twenty.

In [ ]:
# describing the raw_network (buses, lines and loads only) :

In [ ]:
# Describing the DER units (IB, GFL, GFM and SM) and their transformers :
# Which DER is connected to which bus, and what are the parameters of the DER units and their transformers.


In [ ]:
# Complete network description (buses, lines, loads, DER units and transformers) :

In [ ]:
# Validating the network topology and parameters :

In [5]:
# Plotting the network topology :

## 2. Power Flow (PandaPower) : 
pu-base --> power-flow 

In [ ]:
# Computing the pu values of the network elements :

In [ ]:
# Selecting the power flow solver and parameters (PandaPower) :

In [ ]:
# Running the power flow :

In [ ]:
# Power flow results (exposing all the reuslts of the power flow (PandaPower)) :

## 3. Time-series static power-flow simulaion : 
- Simulating a load profile 
- Simulating different dispatch levels for the DERs

In [ ]:
# Simulating a load profile at the different load buses of the network :
# The load profiles at the different load buses :

In [ ]:
# Plotting the load profiles at the different load buses :

In [ ]:
# Run batch power flow (PandaPower) : 

In [ ]:
# Power flow results :

## 4. State-space models and linearization :
- power-flow reuslts --> operating points 
- lineaization
- creating symbolic individual state spaces --> creating the global state-space (symbolic) --> The resulting overall state space after substituting with the parameters and operating points

## 5. Modal Analysis toolbox : 
- Eigenvalue map & modal analysis summary 
- Participation factors (heatmap + single-mode participation)
- Sensitivity heatmap
- Mode shape
- Free motion response
- Forced step response

## 6. EMT time-domain simulations : 
- Choice of the state or input to perturb 
- Choice of the solver and its parameters (timestep, .. )
- Choice of the variables to plot 
- Simulate + popup window of the live-plots
- plots in the notebook 
- comparison with linear model behaviour 

## 7. Region of attraction analysis : 

## 8. Additional analysis : 
- plotting the root locus of the mode map for changing a parameter.
- Simulating a step-response in an input with linear forced response versus EMT time-domain simulations
- Simulating the loss of a DER 
- Simulating the loss of a load